# Generate metamapper test data

This notebook writes deterministic Parquet files to the configured GCS bucket.
It creates 150 valid filenames and 50 deliberately invalid filenames.

In [ ]:
import gcsfs
import pandas as pd

BUCKET = "ssb-play-enhjoern-a-data-produkt-test"

VALID_DATASETS = [
    ("befolkning", "inndata", "befolkning"),
    ("befolkning", "statistikk", "befolkning-kommuner"),
    ("befolkning", "utdata", "befolkning"),
    ("sysselsetting", "inndata", "sysselsetting"),
    ("sysselsetting", "statistikk", "sysselsetting-kjonn"),
    ("sysselsetting", "utdata", "sysselsetting"),
    ("utdanning", "klargjorte-data", "utdanning"),
    ("utdanning", "statistikk", "utdanning-nivaa"),
    ("utdanning", "utdata", "utdanning"),
    ("inntekt", "inndata", "personinntekt"),
    ("inntekt", "statistikk", "personinntekt-kommuner"),
    ("inntekt", "utdata", "personinntekt"),
    ("varehandel", "inndata", "varehandel"),
    ("varehandel", "statistikk", "varehandel-imputert"),
    ("varehandel", "utdata", "varehandel"),
]

PERIODS_AND_VERSIONS = [
    ("2024", 1), ("2025", 1), ("2026", 1),
    ("2025-Q1", 1), ("2025-Q2", 1), ("2025-Q3", 1),
    ("2025-Q4", 1), ("2026-Q1", 2), ("2026-Q2", 2),
    ("2026-Q3", 2),
]

INVALID_FILES = [
    "befolkning/statistikk/befolkning_p2025.parquet",
    "sysselsetting/statistikk/sysselsetting_p2025.parquet",
    "utdanning/statistikk/utdanning_p2025.parquet",
    "inntekt/statistikk/personinntekt_p2025.parquet",
    "varehandel/statistikk/varehandel_p2025.parquet",
    "befolkning/utdata/befolkning_p2026-Q1.parquet",
    "sysselsetting/utdata/sysselsetting_p2026-Q1.parquet",
    "utdanning/utdata/utdanning_p2026-Q1.parquet",
    "inntekt/utdata/personinntekt_p2026-Q1.parquet",
    "varehandel/utdata/varehandel_p2026-Q1.parquet",
    "befolkning/statistikk/befolkning_2025_v1.parquet",
    "sysselsetting/statistikk/sysselsetting-kjonn_2025_v1.parquet",
    "utdanning/statistikk/utdanning-nivaa_2025_v1.parquet",
    "inntekt/statistikk/personinntekt_2025_v1.parquet",
    "varehandel/statistikk/varehandel-imputert_2025_v1.parquet",
    "befolkning/utdata/befolkning_2026-Q1_v1.parquet",
    "sysselsetting/utdata/sysselsetting_2026-Q1_v1.parquet",
    "utdanning/utdata/utdanning_2026-Q1_v1.parquet",
    "inntekt/utdata/personinntekt_2026-Q1_v1.parquet",
    "varehandel/utdata/varehandel_2026-Q1_v1.parquet",
    "befolkning/statistikk/befolkning_p2025_1.parquet",
    "sysselsetting/statistikk/sysselsetting-kjonn_p2025_1.parquet",
    "utdanning/statistikk/utdanning-nivaa_p2025_1.parquet",
    "inntekt/statistikk/personinntekt_p2025_1.parquet",
    "varehandel/statistikk/varehandel-imputert_p2025_1.parquet",
    "befolkning/utdata/befolkning_p2026-Q1_1.parquet",
    "sysselsetting/utdata/sysselsetting_p2026-Q1_1.parquet",
    "utdanning/utdata/utdanning_p2026-Q1_1.parquet",
    "inntekt/utdata/personinntekt_p2026-Q1_1.parquet",
    "varehandel/utdata/varehandel_p2026-Q1_1.parquet",
    "befolkning/statistikk/befolkning_kommuner_p2025_v1.parquet",
    "sysselsetting/statistikk/sysselsetting_kjonn_p2025_v1.parquet",
    "utdanning/statistikk/utdanning_nivaa_p2025_v1.parquet",
    "inntekt/statistikk/personinntekt_kommuner_p2025_v1.parquet",
    "varehandel/statistikk/varehandel_imputert_p2025_v1.parquet",
    "befolkning/utdata/befolkning_kommuner_p2026-Q1_v1.parquet",
    "sysselsetting/utdata/sysselsetting_kjonn_p2026-Q1_v1.parquet",
    "utdanning/utdata/utdanning_nivaa_p2026-Q1_v1.parquet",
    "inntekt/utdata/personinntekt_kommuner_p2026-Q1_v1.parquet",
    "varehandel/utdata/varehandel_imputert_p2026-Q1_v1.parquet",
    "befolkning/data/befolkning_p2025_v1.parquet",
    "sysselsetting/raw/sysselsetting_p2025_v1.parquet",
    "utdanning/statistikkdata/utdanning_p2025_v1.parquet",
    "inntekt/processed/personinntekt_p2025_v1.parquet",
    "varehandel/output/varehandel_p2025_v1.parquet",
    "befolkning/befolkning_p2025_v1.parquet",
    "sysselsetting/sysselsetting_p2025_v1.parquet",
    "utdanning/utdanning_p2025_v1.parquet",
    "inntekt/personinntekt_p2025_v1.parquet",
    "varehandel/varehandel_p2025_v1.parquet",
]

In [ ]:
def create_test_data() -> pd.DataFrame:
    return pd.DataFrame({
        "region": ["0301", "1103", "4601", "5001", "0301"],
        "year": [2024, 2024, 2024, 2024, 2025],
        "value": [100, 250, 175, 320, 125],
    })

def write_parquet(fs: gcsfs.GCSFileSystem, relative_path: str, data: pd.DataFrame) -> None:
    with fs.open(f"{BUCKET}/{relative_path}", "wb") as file:
        data.to_parquet(file, index=False)

In [ ]:
fs = gcsfs.GCSFileSystem()
data = create_test_data()
valid_count = 0
invalid_count = 0

for product, state, description in VALID_DATASETS:
    for period, version in PERIODS_AND_VERSIONS:
        relative_path = f"{product}/{state}/{description}_p{period}_v{version}.parquet"
        write_parquet(fs, relative_path, data)
        valid_count += 1
        print(f"VALID:   {relative_path}")

for relative_path in INVALID_FILES:
    write_parquet(fs, relative_path, data)
    invalid_count += 1
    print(f"INVALID: {relative_path}")

print(f"\nFinished generating test files.")
print(f"Valid files:   {valid_count}")
print(f"Invalid files: {invalid_count}")
print(f"Total files:   {valid_count + invalid_count}")
print(f"Bucket:        gs://{BUCKET}")
assert valid_count == 150
assert invalid_count == 50